# Lab 3 Exercise - Models, Guardrails and Structured Outputs

**Contributor:** Akhila Guska ([@akhilaguska27](https://github.com/akhilaguska27))
**Course:** AI Engineer Agentic Track - The Complete Agent & MCP Course

This notebook covers the Lab 3 exercise:
1. Try different models (using OpenAI gpt-4o-mini with different personas)
2. Add more input and output guardrails
3. Use structured outputs for email generation

---

## What is new in Lab 3 vs Lab 2?

Lab 2 built the multi-agent email pipeline. Lab 3 adds three production-ready features:

**1. Different Models** - Plug in any AI model with an OpenAI-compatible API (DeepSeek, Gemini, Llama)

**2. Structured Outputs** - Force agents to return specific data formats using Pydantic instead of free text

**3. Guardrails** - Safety checks that run BEFORE the main agent to block unsafe or unwanted requests

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
# New imports compared to Lab 2:
# - input_guardrail: decorator that runs a check before the agent
# - GuardrailFunctionOutput: return type for guardrail functions
# - BaseModel: from pydantic, used to define structured output schemas

from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel
import asyncio

In [ ]:
# ── LOAD ENVIRONMENT VARIABLES ───────────────────────────────────────────────
# Loads API keys from .env file
load_dotenv(override=True)

In [ ]:
# ── SSL FIX FOR MAC USERS ────────────────────────────────────────────────────
# Required on Mac to fix SSL certificate errors when sending emails via SendGrid
# Run this cell before any SendGrid calls
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()

In [ ]:
# ── CHECK API KEYS ───────────────────────────────────────────────────────────
# Only OPENAI_API_KEY is required for this notebook
# The others are optional - if you have them, the original lab uses them
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - required!")

---
## Part 1 - Different Models

The original lab uses DeepSeek, Gemini and Llama via Groq.
This version uses gpt-4o-mini for all 3 agents with different personas to demonstrate
that the SAME model can behave very differently based on instructions.

If you have other API keys, swap the model parameter using OpenAIChatCompletionsModel
with the appropriate base_url and api_key.

In [ ]:
# ── SALES AGENT INSTRUCTIONS ─────────────────────────────────────────────────
# Three different writing styles for cold email generation
# Each agent has a distinct personality despite using the same model

instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
# ── THREE SALES AGENTS WITH DIFFERENT PERSONAS ───────────────────────────────
# All use gpt-4o-mini but with different instructions
# In the original lab these used DeepSeek, Gemini and Llama
# To use other models, replace model= with an OpenAIChatCompletionsModel object
#
# Example with DeepSeek (if you have DEEPSEEK_API_KEY):
# deepseek_client = AsyncOpenAI(base_url="https://api.deepseek.com/v1", api_key=os.getenv('DEEPSEEK_API_KEY'))
# deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
# sales_agent1 = Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=deepseek_model)

sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Concise Sales Agent", instructions=instructions3, model="gpt-4o-mini")

In [ ]:
# ── CONVERT AGENTS TO TOOLS ──────────────────────────────────────────────────
# Each sales agent becomes a tool the Sales Manager can call
# as_tool() wraps the agent so another agent can invoke it

description = "Write a cold sales email"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)
tools = [tool1, tool2, tool3]

In [ ]:
# ── EMAIL SENDING TOOL ───────────────────────────────────────────────────────
# Sends HTML email via SendGrid
# Change from_email and to_email to your verified sender and recipient

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to all sales prospects."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("your-verified-sender@gmail.com")  # change to your verified sender
    to_email = To("your-recipient@gmail.com")              # change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
# ── SUBJECT WRITER AND HTML CONVERTER AGENTS ─────────────────────────────────
# These helper agents handle formatting before sending
# subject_writer: writes a compelling email subject
# html_converter: converts plain text to styled HTML email

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

email_tools = [subject_tool, html_tool, send_html_email]

In [ ]:
# ── EMAIL MANAGER AGENT ──────────────────────────────────────────────────────
# Receives the winning email body via handoff from Sales Manager
# Then formats it and sends it

emailer_agent = Agent(
    name="Email Manager",
    instructions="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body.",
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it"
)

In [ ]:
# ── SALES MANAGER INSTRUCTIONS ───────────────────────────────────────────────
# The orchestrator agent that directs all 3 sales agents
# and picks the best email to send

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts.
   Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email.

3. Handoff for Sending: Pass ONLY the winning email draft to the Email Manager agent.

Crucial Rules:
- Use the sales agent tools to generate drafts - do not write them yourself.
- Hand off exactly ONE email to the Email Manager - never more than one.
"""

---
## Part 2 - More Guardrails

The original lab had ONE guardrail that blocked personal names.
This exercise adds TWO more guardrails:
- Guardrail 2: Block competitor mentions
- Guardrail 3: Block inappropriate content

All guardrails run IN PARALLEL before the main agent starts.
If ANY guardrail triggers, the request is blocked immediately.

In [ ]:
# ── STRUCTURED OUTPUTS FOR GUARDRAILS ────────────────────────────────────────
# Guardrail agents need to return True/False decisions
# We use Pydantic BaseModel to force structured output
# This is more reliable than parsing free text

# Original from lab - checks for personal names
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

# New - checks for competitor company mentions
class CompetitorCheckOutput(BaseModel):
    is_competitor_mentioned: bool
    competitor_name: str

# New - checks for inappropriate content
class ProfanityCheckOutput(BaseModel):
    is_inappropriate: bool
    reason: str

In [ ]:
# ── GUARDRAIL AGENTS ─────────────────────────────────────────────────────────
# Each guardrail has its own small agent that does a specific check
# output_type forces structured output using the Pydantic models above

# Original from lab - checks for personal names
guardrail_agent = Agent(
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

# New - checks for competitor company mentions
# ComplAI competitors in SOC2 compliance space
competitor_check_agent = Agent(
    name="Competitor Check",
    instructions="""Check if the user message mentions any competitor companies to ComplAI.
    Competitors include: Drata, Vanta, Secureframe, Tugboat Logic, Sprinto, Laika, or any other SOC2 compliance tools.
    Return True if any competitors are mentioned.""",
    output_type=CompetitorCheckOutput,
    model="gpt-4o-mini"
)

# New - checks for inappropriate or unprofessional content
profanity_check_agent = Agent(
    name="Profanity Check",
    instructions="""Check if the user message contains any inappropriate, offensive, or unprofessional content
    that should not appear in a business sales email.
    Return True if the content is inappropriate for professional communication.""",
    output_type=ProfanityCheckOutput,
    model="gpt-4o-mini"
)

In [ ]:
# ── GUARDRAIL FUNCTIONS ──────────────────────────────────────────────────────
# @input_guardrail decorator marks these as guardrails
# tripwire_triggered=True means BLOCK the request
# tripwire_triggered=False means ALLOW the request
#
# How it works:
# 1. User sends message
# 2. ALL guardrails run in parallel
# 3. If ANY tripwire triggers -> request blocked, InputGuardrailTripwireTriggered raised
# 4. If ALL pass -> main agent runs normally

# Original from lab
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(
        output_info={"found_name": result.final_output},
        tripwire_triggered=is_name_in_message
    )

# New - blocks competitor mentions
@input_guardrail
async def guardrail_against_competitors(ctx, agent, message):
    result = await Runner.run(competitor_check_agent, message, context=ctx.context)
    is_competitor_mentioned = result.final_output.is_competitor_mentioned
    return GuardrailFunctionOutput(
        output_info={"found_competitor": result.final_output},
        tripwire_triggered=is_competitor_mentioned
    )

# New - blocks inappropriate content
@input_guardrail
async def guardrail_against_profanity(ctx, agent, message):
    result = await Runner.run(profanity_check_agent, message, context=ctx.context)
    is_inappropriate = result.final_output.is_inappropriate
    return GuardrailFunctionOutput(
        output_info={"inappropriate": result.final_output},
        tripwire_triggered=is_inappropriate
    )

In [ ]:
# ── SUPER CAREFUL SALES MANAGER ──────────────────────────────────────────────
# Has ALL 3 guardrails applied
# Blocks: personal names, competitor mentions, inappropriate content
# All 3 guardrails run in parallel before the agent starts

super_careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[
        guardrail_against_name,           # blocks personal names
        guardrail_against_competitors,    # blocks competitor mentions
        guardrail_against_profanity       # blocks inappropriate content
    ]
)

In [ ]:
# ── TEST 1: SHOULD PASS ──────────────────────────────────────────────────────
# No personal name, no competitor, no inappropriate content
# All 3 guardrails should pass and the email should be sent

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Multi-guardrail SDR - Safe"):
    result = await Runner.run(super_careful_sales_manager, message)
    print("PASSED - Email sent successfully!")

In [ ]:
# ── TEST 2: SHOULD BE BLOCKED - competitor mention ────────────────────────────
# Mentions Vanta which is a competitor
# guardrail_against_competitors should trigger and block this

try:
    message = "Send out a cold sales email saying we are better than Vanta"
    with trace("Multi-guardrail SDR - Competitor blocked"):
        result = await Runner.run(super_careful_sales_manager, message)
except Exception as e:
    print(f"BLOCKED by competitor guardrail as expected!")
    print(f"Error type: {type(e).__name__}")

In [ ]:
# ── TEST 3: SHOULD BE BLOCKED - personal name ─────────────────────────────────
# Contains personal name Alice
# guardrail_against_name should trigger and block this

try:
    message = "Send out a cold sales email from Alice"
    with trace("Multi-guardrail SDR - Name blocked"):
        result = await Runner.run(super_careful_sales_manager, message)
except Exception as e:
    print(f"BLOCKED by name guardrail as expected!")
    print(f"Error type: {type(e).__name__}")

---
## Part 3 - Structured Outputs for Email Generation

Instead of agents returning plain text, we force them to return
a structured Python object with specific fields.

Benefits:
- No text parsing needed
- Access fields directly like email.subject, email.tone
- Reliable and consistent format every time
- Easy to process programmatically

In [ ]:
# ── STRUCTURED EMAIL OUTPUT SCHEMA ───────────────────────────────────────────
# Defines exactly what fields the agent must return
# Pydantic BaseModel validates the output automatically
# If the agent returns the wrong format, it will be corrected

class EmailOutput(BaseModel):
    subject: str                    # the email subject line
    body: str                       # the full email body text
    tone: str                       # professional / humorous / concise
    estimated_response_rate: str    # High / Medium / Low

In [ ]:
# ── STRUCTURED OUTPUT SALES AGENT ────────────────────────────────────────────
# output_type=EmailOutput forces the agent to return an EmailOutput object
# instead of a plain text string
# The SDK handles the structured output conversion automatically

structured_sales_agent = Agent(
    name="Structured Sales Agent",
    instructions="""You are a sales agent for ComplAI, a SOC2 compliance SaaS tool.
    Write a cold sales email and return it as structured output with:
    - subject: a compelling subject line
    - body: the full email body text
    - tone: describe the tone as one of professional, humorous, or concise
    - estimated_response_rate: your estimate as High, Medium, or Low""",
    output_type=EmailOutput,
    model="gpt-4o-mini"
)

In [ ]:
# ── RUN STRUCTURED OUTPUT AGENT ──────────────────────────────────────────────
# Run the agent and access each field separately
# No text parsing needed - just dot notation!
#
# Compare this to Lab 2 where you got a big text blob
# Now you get a proper Python object with named fields

with trace("Structured email output"):
    result = await Runner.run(
        structured_sales_agent,
        "Write a cold sales email addressed to Dear CEO from Head of Sales"
    )

email = result.final_output

# Access each field directly - clean and reliable!
print(f"Subject: {email.subject}")
print(f"Tone: {email.tone}")
print(f"Estimated Response Rate: {email.estimated_response_rate}")
print(f"\nEmail Body:\n{email.body}")

In [ ]:
# ── COMPARE 3 STRUCTURED EMAILS SIDE BY SIDE ─────────────────────────────────
# Run 3 agents in parallel and compare their structured outputs
# This shows how structured outputs make it easy to compare results
# programmatically instead of reading through walls of text

structured_agent1 = Agent(
    name="Professional Structured Agent",
    instructions="You write professional cold sales emails for ComplAI, a SOC2 compliance SaaS.",
    output_type=EmailOutput,
    model="gpt-4o-mini"
)

structured_agent2 = Agent(
    name="Humorous Structured Agent",
    instructions="You write witty, humorous cold sales emails for ComplAI, a SOC2 compliance SaaS.",
    output_type=EmailOutput,
    model="gpt-4o-mini"
)

structured_agent3 = Agent(
    name="Concise Structured Agent",
    instructions="You write concise, to the point cold sales emails for ComplAI, a SOC2 compliance SaaS.",
    output_type=EmailOutput,
    model="gpt-4o-mini"
)

message = "Write a cold sales email addressed to Dear CEO"

with trace("Structured parallel emails"):
    results = await asyncio.gather(
        Runner.run(structured_agent1, message),
        Runner.run(structured_agent2, message),
        Runner.run(structured_agent3, message),
    )

# Compare all 3 side by side using structured fields
print("=" * 60)
for i, result in enumerate(results, 1):
    email = result.final_output
    print(f"\nEmail {i}:")
    print(f"  Subject: {email.subject}")
    print(f"  Tone: {email.tone}")
    print(f"  Response Rate: {email.estimated_response_rate}")
    print("-" * 60)

---

## Key Learnings from Lab 3

| Concept | What it does | Why it matters |
|---|---|---|
| Different models | Swap any OpenAI-compatible model | Cost optimization, best model for each task |
| Structured outputs | Force agent to return specific format | Reliable data, no text parsing |
| Input guardrails | Safety checks before agent runs | Block misuse, PII, competitors |
| Multiple guardrails | All run in parallel | Fast safety checking at scale |

## Guardrail Results Summary

| Test | Message | Expected | Why |
|---|---|---|---|
| Test 1 | Dear CEO from Head of Sales | PASS | No name, no competitor, no profanity |
| Test 2 | Better than Vanta | BLOCKED | Vanta is a competitor |
| Test 3 | From Alice | BLOCKED | Alice is a personal name |

---
*Contributed by Akhila Guska - feel free to build on this!*